# 5. Conditioning and Classifier-Free Guidance

Two things happen in this notebook, and the second is the more interesting one.

**Conditioning** is easy: restrict the positives *and* the negatives to the conditioning group, so the equilibrium
becomes $q_\theta(\cdot|c)=p_{\text{data}}(\cdot|c)$ for every $c$.

**Guidance** is where drifting does something structurally different from diffusion. Standard classifier-free guidance
extrapolates *predictions* at inference:
$\hat\epsilon = \epsilon_\theta(\cdot|\varnothing) + w\big(\epsilon_\theta(\cdot|c)-\epsilon_\theta(\cdot|\varnothing)\big)$,
which costs **two network evaluations per step**. Drifting instead changes **which negatives it repels from during
training**, so the extrapolated distribution becomes a property of the *equilibrium* — and inference stays at
**1 NFE**, with $\alpha$ selectable at test time.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Conditional drifting, vectorised

The paper computes the field **independently per label**: a sample is attracted only to positives of its own class and
repelled only by generated samples of its own class. Looping over labels in Python is slow. The trick is to build one
big logit matrix and add $-\infty$ wherever the labels differ — both softmaxes then respect the grouping, and the whole
batch is one `cdist`.

In [ ]:
NEG_INF = -1e6

def V_grouped(x, x_grp, y_pos, pos_grp, y_unc=None, log_w=None, tau=1.0):
    """Group-masked drifting field.

    x, x_grp        : [N, D], [N]     generated samples and their conditioning group
    y_pos, pos_grp  : [P, D], [P]     positives and their group
    y_unc, log_w    : [U, D], [N]     optional extra negatives shared across groups (CFG),
                                      with a per-row log kernel weight
    """
    N, P = x.shape[0], y_pos.shape[0]
    same_p = torch.where(x_grp[:, None] == pos_grp[None, :], 0.0, NEG_INF)
    same_n = torch.where(x_grp[:, None] == x_grp[None, :], 0.0, NEG_INF)

    lp = -torch.cdist(x, y_pos) / tau + same_p
    ln = -torch.cdist(x, x) / tau + same_n - torch.eye(N, device=x.device) * 1e6
    cols, negs = [lp, ln], [x]
    if y_unc is not None:
        cols.append(-torch.cdist(x, y_unc) / tau + log_w[:, None])   # shared across groups
        negs.append(y_unc)

    logit = torch.cat(cols, 1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2) + 1e-30)
    A_pos, A_neg = A[:, :P], A[:, P:]
    Yn = torch.cat(negs, 0)
    return (A_pos * A_neg.sum(1, keepdim=True)) @ y_pos - (A_neg * A_pos.sum(1, keepdim=True)) @ Yn

def drift_loss(f, f_pos, pos_grp, x_grp, f_unc=None, log_w=None,
               taus=(0.02, 0.05, 0.2), eta=0.05):
    C = f.shape[-1]
    with torch.no_grad():
        S = (torch.cdist(f, f_pos).mean() / math.sqrt(C)).clamp_min(1e-8)
    fn, fpn = f / S, f_pos / S
    fun = None if f_unc is None else f_unc / S
    with torch.no_grad():
        V = 0.0
        for t in taus:
            v = V_grouped(fn, x_grp, fpn, pos_grp, fun, log_w, tau=t * math.sqrt(C))
            V = V + v / v.pow(2).mean().sqrt().clamp_min(1e-8)
    return F.mse_loss(fn, (fn + eta * V).detach())

## 2. A 2-D sanity check

Four classes, each a rotated crescent. If conditioning works, sampling with label $c$ should produce only crescent $c$.

In [ ]:
def crescents(n, n_cls=4, device=device):
    c = torch.randint(0, n_cls, (n,), device=device)
    ang = c.float() * (2 * math.pi / n_cls)
    t = torch.rand(n, device=device) * math.pi - math.pi / 2
    r = 0.75 + 0.06 * torch.randn(n, device=device)
    x = torch.stack([r * torch.cos(t), r * torch.sin(t)], -1)
    ca, sa = torch.cos(ang), torch.sin(ang)
    R = torch.stack([torch.stack([ca, -sa], -1), torch.stack([sa, ca], -1)], -2)
    return torch.einsum("nij,nj->ni", R, x), c

class CondGen(nn.Module):
    def __init__(self, d_z=8, d_out=2, n_cls=4, h=256, nl=4, d_emb=32):
        super().__init__()
        self.emb = nn.Embedding(n_cls, d_emb)
        self.alpha = nn.Linear(1, d_emb)
        L, d = [], d_z + 2 * d_emb
        for _ in range(nl):
            L += [nn.Linear(d, h), nn.SiLU()]; d = h
        L += [nn.Linear(d, d_out)]
        self.net = nn.Sequential(*L); self.d_z = d_z
    def forward(self, z, c, a):
        return self.net(torch.cat([z, self.emb(c), self.alpha(a[:, None])], -1))

def train_cond(steps=3000, n_cls=4, N_grp=8, N_neg=32, N_pos=32, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    g = CondGen(n_cls=n_cls).to(device)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95), weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    pool_x, pool_c = crescents(40000, n_cls)
    by_cls = [torch.where(pool_c == k)[0] for k in range(n_cls)]
    for s in range(steps):
        grp = torch.randint(0, n_cls, (N_grp,), device=device)
        x_grp = grp.repeat_interleave(N_neg)
        ones = torch.ones(x_grp.shape[0], device=device)
        x = g(torch.randn(x_grp.shape[0], g.d_z, device=device), x_grp, ones)
        idx = torch.cat([by_cls[k][torch.randint(0, len(by_cls[k]), (N_pos,), device=device)]
                         for k in grp.tolist()])
        y_pos, pos_grp = pool_x[idx], grp.repeat_interleave(N_pos)
        loss = drift_loss(x, y_pos, pos_grp, x_grp)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step(); sched.step()
    return g

g4 = train_cond()

real_x, real_c = crescents(4000)
fig, ax = plt.subplots(1, 5, figsize=(15, 3.1))
cols = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
for k in range(4):
    m = real_c == k
    ax[0].scatter(real_x[m, 0].cpu(), real_x[m, 1].cpu(), s=2, c=cols[k], alpha=0.35)
ax[0].set_title("p_data (4 classes)", fontsize=9)
for k in range(4):
    with torch.no_grad():
        s = g4(torch.randn(1500, g4.d_z, device=device),
               torch.full((1500,), k, device=device),
               torch.ones(1500, device=device))
    ax[k + 1].scatter(real_x[:, 0].cpu(), real_x[:, 1].cpu(), s=1.5, c="lightgray")
    ax[k + 1].scatter(s[:, 0].cpu(), s[:, 1].cpu(), s=2, c=cols[k], alpha=0.4)
    ax[k + 1].set_title(f"sampled with c = {k}", fontsize=9)
for a in ax:
    a.set_aspect("equal"); a.set_xlim(-1.4, 1.4); a.set_ylim(-1.4, 1.4)
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

Clean separation. Note there is nothing special in the architecture — a label embedding concatenated to the noise. The
conditioning lives in the **field**, via the group mask.

## 3. Guidance as a negative mixture

Here is the derivation (paper §3.5). Keep the positives conditional, $\mathbf{y}^{+}\sim p_{\text{data}}(\cdot|c)$, but
make the *negative* distribution a mixture of your own samples and **real unconditional data**:

$$\tilde q(\cdot|c) \;=\; (1-\gamma)\,q_\theta(\cdot|c) \;+\; \gamma\,p_{\text{data}}(\cdot|\varnothing),
\qquad \gamma\in[0,1)$$

Equilibrium is when the *effective negative* matches the positive, $\tilde q(\cdot|c)=p_{\text{data}}(\cdot|c)$.
Solve for what the model itself must then be:

$$\boxed{\;q_\theta(\cdot|c) \;=\; \alpha\,p_{\text{data}}(\cdot|c) \;-\; (\alpha-1)\,p_{\text{data}}(\cdot|\varnothing),
\qquad \alpha=\tfrac{1}{1-\gamma}\ge 1\;}$$

The generator is trained to match a **linear extrapolation of the conditional away from the unconditional** — exactly
what CFG does, but as an equilibrium property rather than an inference-time formula. $\alpha=1$ recovers plain
conditional generation.

Intuition: you are telling the model *"do not merely look like this class, look like this class and specifically not
like generic data"*. Because the extra negatives are real images from other classes, the pressure is toward the
features that distinguish the class.

### Implementation

Add $N_{\text{unc}}$ real samples from random classes as extra negatives, weighted by $w$ in the kernel. Then the
effective negative mixture is

$$\tilde q(\cdot|c) = \frac{(N_{\text{neg}}-1)\,q_\theta(\cdot|c) + N_{\text{unc}}\,w\,p_{\text{data}}(\cdot|\varnothing)}
{(N_{\text{neg}}-1) + N_{\text{unc}}\,w}$$

$$\gamma = \frac{N_{\text{unc}}w}{(N_{\text{neg}}-1)+N_{\text{unc}}w},
\qquad \alpha = \frac{1}{1-\gamma} = \frac{(N_{\text{neg}}-1)+N_{\text{unc}}w}{N_{\text{neg}}-1}$$

so given a desired $\alpha$ you solve for the weight

$$w = \frac{(\alpha-1)(N_{\text{neg}}-1)}{N_{\text{unc}}}$$

($N_{\text{neg}}-1$ because the sample's own slot is masked out.) The weight enters as `+log(w)` on the logits of the
unconditional block, and the same weighting is applied when computing the distance normalisation.

### CFG-conditioning

The last trick (from improved MeanFlow, adopted by the paper): don't fix $\alpha$. Sample it per group each step,
build the negatives accordingly, and **feed $\alpha$ to the network as a conditioning input**,
$f_\theta(\boldsymbol{\epsilon}, c, \alpha)$. One model then covers a whole family of extrapolations, and at inference
you pick $\alpha$ freely — still one forward pass.

The paper samples $\alpha\in[1,4]$ with $p(\alpha)\propto\alpha^{-3}$ or $\alpha^{-5}$, i.e. heavily biased toward
small $\alpha$, which is where the FID optimum lives.

In [ ]:
def w_from_alpha(alpha, N_neg, N_unc):
    return ((alpha - 1.0) * (N_neg - 1) / N_unc).clamp_min(1e-8)

N_neg, N_unc = 24, 24
a = torch.linspace(1.0, 4.0, 7)
print(f"{'alpha':>6s} {'gamma':>8s} {'w':>8s}   interpretation")
for av in a:
    w = w_from_alpha(av, N_neg, N_unc)
    gam = 1 - 1 / av
    print(f"{av:6.2f} {gam:8.3f} {w:8.3f}   q = {av:.2f} p(.|c) - {av-1:.2f} p(.|uncond)")

## 4. Class-conditional MNIST with CFG-conditioning

Now the real thing: the feature-space encoder from Notebook 4, class conditioning by group masking, and
CFG-conditioning with $\alpha$ sampled per group. Roughly 3 minutes on a GPU.

In [ ]:
ds = torchvision.datasets.MNIST("./data", train=True, download=True)
X = ((ds.data.float() / 255.0) * 2 - 1).to(device)
Yl = ds.targets.to(device)
by_class = [torch.where(Yl == c)[0] for c in range(10)]

class Encoder(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(1, c, 3, 1, 1), nn.GroupNorm(8, c), nn.SiLU(),
                                nn.Conv2d(c, c, 3, 2, 1), nn.GroupNorm(8, c), nn.SiLU())
        self.b2 = nn.Sequential(nn.Conv2d(c, 2*c, 3, 1, 1), nn.GroupNorm(8, 2*c), nn.SiLU(),
                                nn.Conv2d(2*c, 2*c, 3, 2, 1), nn.GroupNorm(8, 2*c), nn.SiLU())
        self.b3 = nn.Sequential(nn.Conv2d(2*c, 4*c, 3, 1, 1), nn.GroupNorm(8, 4*c), nn.SiLU())
    def forward(self, x):
        f1 = self.b1(x); f2 = self.b2(f1); return f1, f2, self.b3(f2)

class Decoder(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(4*c, 2*c, 4, 2, 1), nn.GroupNorm(8, 2*c), nn.SiLU(),
            nn.ConvTranspose2d(2*c, c, 4, 2, 1), nn.GroupNorm(8, c), nn.SiLU(),
            nn.Conv2d(c, 1, 3, 1, 1))
    def forward(self, f): return self.net(f)

def train_mae(steps=3000, N=256):
    torch.manual_seed(0)
    e, d = Encoder().to(device), Decoder().to(device)
    opt = torch.optim.AdamW(list(e.parameters()) + list(d.parameters()), 2e-3)
    for _ in range(steps):
        i = torch.randint(0, X.shape[0], (N,), device=device)
        x = X[i].view(N, 1, 28, 28)
        m = F.interpolate((torch.rand(N, 1, 14, 14, device=device) > 0.5).float(), scale_factor=2)
        loss = (((d(e(x * m)[2]) - x) ** 2) * (1 - m)).sum() / ((1 - m).sum() + 1e-6)
        opt.zero_grad(); loss.backward(); opt.step()
    for p in list(e.parameters()) + list(d.parameters()):
        p.requires_grad_(False)
    print(f"masked AE final loss {loss.item():.4f}")
    return e

enc = train_mae()

def pooled_stats(f, k):
    B, C, H, W = f.shape
    if H % k or W % k:
        return None
    p = f.view(B, C, H//k, k, W//k, k).permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C, k*k)
    return p.mean(-1), p.std(-1)

def feature_set(img):
    B = img.shape[0]
    out = [(img.view(B, 1, -1) ** 2).mean(-1)]
    for f in enc(img):
        out += [f.mean((2, 3)), f.std((2, 3)), f.permute(0, 2, 3, 1).reshape(B, -1)]
        r = pooled_stats(f, 2)
        if r is not None:
            out += [r[0].reshape(B, -1), r[1].reshape(B, -1)]
    return out

In [ ]:
class MnistCondGen(nn.Module):
    def __init__(self, d_z=64, h=1024, nl=4, d_emb=64):
        super().__init__()
        self.cemb = nn.Embedding(10, d_emb)
        self.aproj = nn.Linear(1, d_emb)
        L, d = [], d_z + 2 * d_emb
        for _ in range(nl):
            L += [nn.Linear(d, h), nn.SiLU()]; d = h
        L += [nn.Linear(d, 784)]
        self.net = nn.Sequential(*L); self.d_z = d_z
    def forward(self, z, c, a):
        return torch.tanh(self.net(torch.cat([z, self.cemb(c), self.aproj(a[:, None])], -1)))
    @torch.no_grad()
    def sample(self, n, c, alpha):
        cc = torch.full((n,), c, device=device)
        aa = torch.full((n,), float(alpha), device=device)
        return self(torch.randn(n, self.d_z, device=device), cc, aa)

def train_mnist(steps=8000, N_cls=10, N_neg=24, N_pos=24, N_unc=24, lr=2e-4,
                use_cfg=True, seed=0):
    torch.manual_seed(seed)
    g = MnistCondGen().to(device)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95), weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    t0 = time.time()
    for s in range(steps):
        cls = torch.randint(0, 10, (N_cls,), device=device)
        # p(alpha) heavily biased toward small alpha, as in the paper
        alpha = 1.0 + 3.0 * torch.rand(N_cls, device=device) ** 3 if use_cfg \
                else torch.ones(N_cls, device=device)
        x_grp = cls.repeat_interleave(N_neg)
        a_grp = alpha.repeat_interleave(N_neg)
        x = g(torch.randn(x_grp.shape[0], g.d_z, device=device), x_grp, a_grp)

        idx = torch.cat([by_class[c][torch.randint(0, len(by_class[c]), (N_pos,), device=device)]
                         for c in cls.tolist()])
        y_pos, pos_grp = X[idx].view(-1, 784), cls.repeat_interleave(N_pos)

        y_unc = log_w = None
        if use_cfg:
            iu = torch.randint(0, X.shape[0], (N_unc,), device=device)
            y_unc = X[iu].view(-1, 784)
            log_w = torch.log(w_from_alpha(a_grp, N_neg, N_unc))

        fx = feature_set(x.view(-1, 1, 28, 28))
        with torch.no_grad():
            fy = feature_set(y_pos.view(-1, 1, 28, 28))
            fu = None if y_unc is None else feature_set(y_unc.view(-1, 1, 28, 28))

        loss = drift_loss(x, y_pos, pos_grp, x_grp, y_unc, log_w)
        for k in range(len(fx)):
            loss = loss + drift_loss(fx[k], fy[k], pos_grp, x_grp,
                                     None if fu is None else fu[k], log_w)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), 2.0)
        opt.step(); sched.step()
        if s % 2000 == 0:
            print(f"  step {s:5d}  loss {loss.item():.3e}  ({time.time()-t0:.0f}s)")
    print(f"trained in {time.time()-t0:.0f}s")
    return g

gm = train_mnist()

In [ ]:
fig, ax = plt.subplots(4, 10, figsize=(13, 5.4))
for r, al in enumerate([1.0, 1.5, 2.5, 4.0]):
    for c in range(10):
        ax[r, c].imshow(gm.sample(1, c, al).view(28, 28).cpu(), cmap="gray")
        ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
        if r == 0:
            ax[r, c].set_title(str(c), fontsize=8)
    ax[r, 0].set_ylabel(f"alpha={al}", fontsize=8)
plt.suptitle("one model, one forward pass per sample, alpha chosen at inference", y=1.01, fontsize=10)
plt.tight_layout(); plt.show()

## 5. Measuring the guidance trade-off

In the paper, raising $\alpha$ raises Inception Score (per-image quality / class fidelity) and eventually hurts FID
(distributional coverage). We can measure the same trade-off directly on MNIST with a small classifier:

* **class fidelity** — does a classifier agree with the requested label, and how confident is it?
* **coverage** — the spread of the generated samples for a fixed class (intra-class diversity).

In [ ]:
class Clf(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, c, 3, 1, 1), nn.SiLU(), nn.MaxPool2d(2),
                               nn.Conv2d(c, 2*c, 3, 1, 1), nn.SiLU(), nn.MaxPool2d(2),
                               nn.Flatten(), nn.Linear(2*c*49, 128), nn.SiLU(), nn.Linear(128, 10))
    def forward(self, x): return self.f(x)

torch.manual_seed(0)
clf = Clf().to(device)
o = torch.optim.Adam(clf.parameters(), 2e-3)
for _ in range(1200):
    i = torch.randint(0, X.shape[0], (256,), device=device)
    loss = F.cross_entropy(clf(X[i].view(-1, 1, 28, 28)), Yl[i])
    o.zero_grad(); loss.backward(); o.step()
with torch.no_grad():
    i = torch.randperm(X.shape[0], device=device)[:5000]
    print(f"classifier accuracy on real data: {(clf(X[i].view(-1,1,28,28)).argmax(-1)==Yl[i]).float().mean():.3%}")
for p in clf.parameters():
    p.requires_grad_(False)

In [ ]:
@torch.no_grad()
def cfg_metrics(g, alpha, n_per_class=200):
    accs, confs, divs = [], [], []
    for c in range(10):
        s = g.sample(n_per_class, c, alpha)
        logits = clf(s.view(-1, 1, 28, 28))
        pr = logits.softmax(-1)
        accs.append((logits.argmax(-1) == c).float().mean().item())
        confs.append(pr.max(-1).values.mean().item())
        divs.append(torch.cdist(s, s).mean().item())      # mean pairwise distance within class
    return float(np.mean(accs)), float(np.mean(confs)), float(np.mean(divs))

alphas = [1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
rows = [cfg_metrics(gm, a) for a in alphas]
acc = [r[0] for r in rows]; conf = [r[1] for r in rows]; div = [r[2] for r in rows]

# reference: real data
real_div = float(np.mean([torch.cdist(X[by_class[c][:200]].view(-1, 784),
                                      X[by_class[c][:200]].view(-1, 784)).mean().item()
                          for c in range(10)]))

print(f"{'alpha':>6s} {'class acc':>10s} {'confidence':>11s} {'intra-class spread':>19s}")
for a, ac, cf, dv in zip(alphas, acc, conf, div):
    print(f"{a:6.2f} {ac:10.3f} {cf:11.3f} {dv:19.2f}")
print(f"{'real':>6s} {'-':>10s} {'-':>11s} {real_div:19.2f}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(alphas, acc, "-o"); ax[0].set_xlabel("alpha"); ax[0].set_title("class fidelity (classifier acc)")
ax[1].plot(alphas, conf, "-o", c="tab:orange"); ax[1].set_xlabel("alpha"); ax[1].set_title("classifier confidence")
ax[2].plot(alphas, div, "-o", c="tab:green"); ax[2].axhline(real_div, ls="--", c="k", label="real data")
ax[2].set_xlabel("alpha"); ax[2].set_title("intra-class spread (coverage)"); ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

This is the classic guidance trade-off, obtained without a second forward pass and with $\alpha$ selected at inference
time. What you should see (exact numbers vary with seed and training budget) is fidelity/confidence rising with
$\alpha$ while intra-class spread falls away from the real-data line — the same shape as the paper's Figure 5, where
FID has a sweet spot at small $\alpha$ while IS keeps climbing.

One curiosity worth knowing: the paper's **best** model has its FID optimum at $\alpha=1.0$, which in
diffusion terms means "no CFG". That does not mean the mechanism is idle — training still used real unconditional
samples as negatives, which shapes the equilibrium. It means the extrapolation strength that helps a weaker model
is unnecessary for a strong one.

## 6. Why this is a genuinely different mechanism

| | diffusion CFG | drifting guidance |
|---|---|---|
| where it acts | inference, on predictions | training, on the negative distribution |
| cost at inference | **2 NFE per step** | **1 NFE, total** |
| what is extrapolated | the score / velocity estimate | the equilibrium distribution itself |
| needs an unconditional model | yes (label dropout during training) | no — uses *real* unconditional samples as negatives |
| $\alpha$ selectable at test time | yes | yes (via CFG-conditioning) |

The conceptual point: in diffusion, guidance is a *hack applied to a trained model's outputs*. In drifting, guidance is
a *specification of what the trained model should be* — equation (45) of `THEORY.md` says precisely which distribution
the equilibrium encodes. That is a cleaner object, and it composes with anything else you can express as a change to
the negative pool. Notebook 10 uses exactly this freedom to repel from a *toxicity* distribution instead of an
unconditional one.

## Summary

* Conditioning = restrict positives and negatives to the group. Vectorise with an $-\infty$ label mask.
* Guidance = mix real unconditional samples into the negative distribution. The equilibrium then encodes
  $q = \alpha\,p(\cdot|c) - (\alpha-1)\,p(\cdot|\varnothing)$.
* Convert a target $\alpha$ into a kernel weight $w=(\alpha-1)(N_{\text{neg}}-1)/N_{\text{unc}}$.
* CFG-condition on $\alpha$ so one model spans the whole family; inference stays 1 NFE.
* The fidelity/coverage trade-off behaves like ordinary CFG, measured here with a classifier.

Next: [6. Drifting vs Diffusion, Flow, GAN and MMD](<6. Drifting vs Diffusion, Flow, GAN and MMD.ipynb>).